# Do card payers pay higher taxi fares? Separating tips and trip length

**Course exercise, revised.** Self-initiated re-analysis of a course project. The business scenario is fictitious and the course provider altered the data for teaching. This is not client work. Revised with AI assistance.

This is **observational** data: riders chose how to pay. Nobody was randomly assigned a payment method, so the results describe associations, not effects.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats

df = pd.read_csv("2017_Yellow_Taxi_Trip_Data.csv", index_col=0)
print(f"Rows in file: {len(df):,}")

Rows in file: 22,699


## 1. Prepare card and cash trips
`payment_type` 1 is credit card and 2 is cash. The [TLC data dictionary](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf) states that tips are recorded for card payments only and that `total_amount` excludes cash tips.

In [2]:
# payment_type 1 = credit card, 2 = cash
d = df[df.payment_type.isin([1, 2])].copy()
print(f"Card or cash trips: {len(d):,}")
d["card"] = (d.payment_type == 1).astype(int)
d["pickup"] = pd.to_datetime(d.tpep_pickup_datetime, format="%m/%d/%Y %I:%M:%S %p")
d["dropoff"] = pd.to_datetime(d.tpep_dropoff_datetime, format="%m/%d/%Y %I:%M:%S %p")
d["minutes"] = (d.dropoff - d.pickup).dt.total_seconds() / 60

Card or cash trips: 22,532


## 2. Data quality checks

In [3]:
print("\nData quality flags (card and cash trips)")
flags = {
    "fare <= 0": d.fare_amount <= 0,
    "distance <= 0": d.trip_distance <= 0,
    "duration <= 0": d.minutes <= 0,
    "duration > 180 min": d.minutes > 180,
    "fare > 200": d.fare_amount > 200,
    "cash trip with a recorded tip": (d.card == 0) & (d.tip_amount > 0),
}
for name, mask in flags.items():
    print(f"  {name}: {int(mask.sum())}")


Data quality flags (card and cash trips)
  fare <= 0: 4
  distance <= 0: 125
  duration <= 0: 25
  duration > 180 min: 45
  fare > 200: 3
  cash trip with a recorded tip: 0


Trips with impossible or extreme values are removed. The thresholds are analyst choices, so unfiltered results are shown as well.

In [4]:
bad = (d.fare_amount <= 0) | (d.trip_distance <= 0) | (d.minutes <= 0) | (d.minutes > 180) | (d.fare_amount > 200)
c = d[~bad].copy()
print(f"Trips kept after filtering: {len(c):,} (removed {int(bad.sum()):,})")

Trips kept after filtering: 22,359 (removed 173)


## 3. Welch two-sample comparisons (card minus cash, USD)

In [5]:
def compare(data, col, label):
    a, b = data.loc[data.card == 1, col], data.loc[data.card == 0, col]
    res = stats.ttest_ind(a, b, equal_var=False)
    se2a, se2b = a.var(ddof=1) / len(a), b.var(ddof=1) / len(b)
    se = np.sqrt(se2a + se2b)
    dof = (se2a + se2b) ** 2 / (se2a**2 / (len(a) - 1) + se2b**2 / (len(b) - 1))
    diff = a.mean() - b.mean()
    half = stats.t.ppf(0.975, dof) * se
    pooled = np.sqrt(((len(a) - 1) * a.var() + (len(b) - 1) * b.var()) / (len(a) + len(b) - 2))
    print(f"{label:38s} card {a.mean():6.2f}  cash {b.mean():6.2f}  diff {diff:5.2f} "
          f"[95% CI {diff - half:5.2f}, {diff + half:5.2f}]  d={diff / pooled:4.2f}  "
          f"t={res.statistic:5.2f}  p={res.pvalue:.2g}  n={len(a):,}/{len(b):,}")

In [6]:
print("\nWelch comparisons, USD")
compare(d, "total_amount", "total_amount, unfiltered (original)")
compare(d, "fare_amount", "fare_amount, unfiltered")
compare(c, "total_amount", "total_amount, filtered")
compare(c, "fare_amount", "fare_amount, filtered")
compare(c, "trip_distance", "trip_distance (miles), filtered")


Welch comparisons, USD


total_amount, unfiltered (original)    card  17.66  cash  13.55  diff  4.12 [95% CI  3.72,  4.51]  d=0.26  t=20.35  p=4.5e-91  n=15,265/7,267
fare_amount, unfiltered                card  13.43  cash  12.21  diff  1.22 [95% CI  0.87,  1.56]  d=0.09  t= 6.87  p=6.8e-12  n=15,265/7,267
total_amount, filtered                 card  17.44  cash  13.51  diff  3.94 [95% CI  3.59,  4.28]  d=0.30  t=22.56  p=4.2e-111  n=15,181/7,178
fare_amount, filtered                  card  13.25  cash  12.17  diff  1.08 [95% CI  0.78,  1.37]  d=0.10  t= 7.11  p=1.2e-12  n=15,181/7,178
trip_distance (miles), filtered        card   3.04  cash   2.67  diff  0.37 [95% CI  0.27,  0.47]  d=0.10  t= 7.30  p=3.1e-13  n=15,181/7,178


The gap in `total_amount` is much larger than the gap in the metered fare. `total_amount` is not a fair comparison because it contains card tips but no cash tips.

In [7]:
tip_gap = c.loc[c.card == 1, "tip_amount"].mean() - c.loc[c.card == 0, "tip_amount"].mean()
total_gap = c.loc[c.card == 1, "total_amount"].mean() - c.loc[c.card == 0, "total_amount"].mean()
print(f"\nRecorded tips account for {tip_gap:.2f} of the {total_gap:.2f} filtered total_amount gap "
      f"({tip_gap / total_gap:.0%}).")


Recorded tips account for 2.70 of the 3.94 filtered total_amount gap (69%).


## 4. Does the fare gap remain for trips of the same length?

In [8]:
print("\nOLS on fare_amount, HC3 robust standard errors (association, not causation)")
for formula in ["fare_amount ~ card",
                "fare_amount ~ card + trip_distance",
                "fare_amount ~ card + trip_distance + minutes + C(RatecodeID)"]:
    m = smf.ols(formula, data=c).fit(cov_type="HC3")
    lo, hi = m.conf_int().loc["card"]
    print(f"  {formula:62s} card={m.params['card']:6.3f} [{lo:6.3f}, {hi:6.3f}] "
          f"p={m.pvalues['card']:.2g} R2={m.rsquared:.3f}")


OLS on fare_amount, HC3 robust standard errors (association, not causation)
  fare_amount ~ card                                             card= 1.077 [ 0.780,  1.373] p=1.1e-12 R2=0.002
  fare_amount ~ card + trip_distance                             card= 0.049 [-0.055,  0.152] p=0.36 R2=0.879


  fare_amount ~ card + trip_distance + minutes + C(RatecodeID)   card= 0.050 [-0.017,  0.118] p=0.14 R2=0.952


## Conclusions
- Card trips show a total amount USD 3.94 higher (95% interval 3.59 to 4.28). Recorded tips account for USD 2.70 of this (69%).
- The metered fare gap is USD 1.08 (0.78 to 1.37), and card trips are longer on average (3.04 vs 2.67 miles).
- Holding trip distance constant, the fare gap is USD 0.05 (-0.06 to 0.15): no clear difference.

**Business reading:** this data gives no support for promoting card payment as a way to raise fare revenue. Testing that would need a designed experiment and a tip measure covering both payment types.

**Limitations:** observational data; cash tips are unobserved; altered teaching sample, so results should not be generalised to New York City; filter thresholds are analyst choices.

## Revision note
The July 2023 version described a randomised experiment that did not take place, compared `total_amount` (which includes card tips only), recommended encouraging card payment, and stated both that the null was rejected and that there was no significant difference. This version replaces it.